# 📊 02. 탐색적 데이터 분석 (EDA)

## 목적
리뷰 메타데이터를 활용하여 유저 행동 패턴을 파악한다.

## 이 노트북의 결과물
- 월별 긍정/부정 비율 추이 차트
- 플레이타임 구간별 분석 결과
- 언어별 분포 분석
- 얼리액세스 vs 정식출시 비교
- Steam Deck 유저 분석

---
## 1. 라이브러리 & DB 연결

In [2]:
import sys
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px

sys.path.append('..')
from config import DB_PATH, EVENTS

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)

print(f"✅ DB 연결 완료: {db_path}")

✅ DB 연결 완료: ../data/dave_diver.db


In [3]:
df = pd.read_sql("SELECT * FROM reviews", conn)

print(f"총 리뷰 수 : {len(df):,}건")
print(f"컬럼 수    : {df.shape[1]}개")
df.head(3)

총 리뷰 수 : 145,877건
컬럼 수    : 54개


,review_id,language,appid,review_text,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,hardware.adapter_description,hardware.driver_version,hardware.driver_date,hardware.vram_size,timestamp_dev_responded,developer_response,review_date,review_month,playtime_hours_forever,playtime_hours_at_review
0,218669790,latam,1868140,Uno de los juegos más hermosos y cautivadores ...,1771469881,1771469881,1,0,0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-19,2026-02,25.3,24.9
1,218667160,koreana,1868140,2시간만 하고 환불 할려고 했는데 정신 차리니까 16시간 째 하고 있는 게임,1771466551,1771466551,1,0,0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-19,2026-02,17.5,17.5
2,218667034,english,1868140,Sweet! Chum!,1771466393,1771466393,1,0,0,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-19,2026-02,6.3,6.2


---
## 2. 기본 통계 요약

In [4]:
# 전체 긍정률, 평균 플레이타임 등 기본 지표
# 

---
## 3. 월별 긍정/부정 추이

In [5]:
monthly = pd.read_sql("""
    SELECT
        review_month,
        COUNT(*) as total_reviews,
        SUM(voted_up) as positive,
        ROUND(AVG(voted_up) * 100, 1) as positive_rate
    FROM reviews
    GROUP BY review_month
    ORDER BY review_month
""", conn)

monthly.head()

,review_month,total_reviews,positive,positive_rate
0,2022-10,707,684,96.7
1,2022-11,4376,4311,98.5
2,2022-12,1391,1365,98.1
3,2023-01,1151,1124,97.7
4,2023-02,615,608,98.9


In [6]:
# 월별 추이 차트 + EVENTS 세로선 표시
# 

---
## 4. 플레이타임 구간별 분석

In [7]:
playtime = pd.read_sql("""
    SELECT
        CASE
            WHEN playtime_at_review < 120  THEN '1_casual (<2h)'
            WHEN playtime_at_review < 600  THEN '2_regular (2-10h)'
            WHEN playtime_at_review < 3000 THEN '3_engaged (10-50h)'
            ELSE '4_hardcore (50h+)'
        END as segment,
        COUNT(*) as count,
        ROUND(AVG(voted_up) * 100, 1) as positive_rate
    FROM reviews
    GROUP BY segment
    ORDER BY segment
""", conn)

playtime

,segment,count,positive_rate
0,1_casual (<2h),4948,87.8
1,2_regular (2-10h),40411,97.5
2,3_engaged (10-50h),80024,96.7
3,4_hardcore (50h+),20494,98.1


In [8]:
# 구간별 차트
# 

---
## 5. 언어별 분포

In [9]:
language = pd.read_sql("""
    SELECT
        language,
        COUNT(*) as total,
        ROUND(AVG(voted_up) * 100, 1) as positive_rate
    FROM reviews
    GROUP BY language
    ORDER BY total DESC
    LIMIT 15
""", conn)

language

,language,total,positive_rate
0,schinese,54059,96.1
1,english,51732,96.7
2,koreana,11318,97.2
3,tchinese,6690,98.6
4,brazilian,4452,99.5
5,spanish,3925,99.1
6,german,3641,97.7
7,french,1989,97.3
8,russian,1269,95.9
9,turkish,1144,97.3


In [10]:
# 언어별 차트
# 

---
## 6. 얼리액세스 vs 정식출시

In [11]:
early = pd.read_sql("""
    SELECT
        written_during_early_access,
        COUNT(*) as count,
        ROUND(AVG(voted_up) * 100, 1) as positive_rate
    FROM reviews
    GROUP BY written_during_early_access
""", conn)

early

,written_during_early_access,count,positive_rate
0,0,133882,96.7
1,1,11995,98.1


---
## 7. Steam Deck 유저 분석

In [12]:
deck = pd.read_sql("""
    SELECT
        primarily_steam_deck,
        COUNT(*) as count,
        ROUND(AVG(voted_up) * 100, 1) as positive_rate,
        ROUND(AVG(playtime_at_review / 60.0), 1) as avg_hours
    FROM reviews
    GROUP BY primarily_steam_deck
""", conn)

deck

,primarily_steam_deck,count,positive_rate,avg_hours
0,0,139649,96.9,27.4
1,1,6228,95.1,23.9


---
## 8. DB 연결 종료

In [13]:
conn.close()
print("✅ EDA 완료. 다음 단계: 03_nlp_preprocessing.ipynb")

✅ EDA 완료. 다음 단계: 03_nlp_preprocessing.ipynb
